In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
 #custom reference to builtins as per lab manual guidelines
import builtins 
import matplotlib.pyplot as plt

#set random seed for reproducibility of simulation results
np.random.seed(42)

print("SIMULATING HDFS CLUSTER VULNERABILITY")
print("Week 4: Fault Tolerance & Disaster Scenarios\n")

class HDFSFaultToleranceSimulator:
    def __init__(self, num_nodes=10, replication_factor=3):
        self.replication_factor = replication_factor
        self.num_nodes = num_nodes
        self.namenode = {
            #block-to-datanode metadata mapping
            'blocks': {}  
        }
        #initialize 10 dataNodes setup as requested in the assignment
        self.datanodes = {
            f'datanode_{i}': {'blocks': {}, 'status': 'active'}
            for i in range(num_nodes)
        }
        
    def place_dataset_blocks(self, num_blocks=1000):
        """Simulates random block placement strategy across available DataNodes (Page 5 style)"""
        available_nodes = list(self.datanodes.keys())
        
        for i in range(num_blocks):
            block_id = f"blk_{i}"
            
            #select unique nodes for replication factors without replacement
            selected_nodes = np.random.choice(available_nodes, 
                                              self.replication_factor, 
                                              replace=False)
            
            #store block reference inside selected sataNodes
            for node in selected_nodes:
                self.datanodes[node]['blocks'][block_id] = True
                
            #update nameNode central metadata repository
            self.namenode['blocks'][block_id] = {
                'replicas': list(selected_nodes)
            }
            
    def reset_cluster_health(self):
        """Resets the state of all cluster DataNodes back to active before running a new scenario"""
        for node in self.datanodes:
            self.datanodes[node]['status'] = 'active'

    def simulate_disaster_scenarios(self, num_trials=1000, failure_rate=0.05):
        """Simulates 1000 independent disaster scenarios and counts total data loss events"""
        trials_lost_blocks = []
        node_names = list(self.datanodes.keys())
        
        for trial in range(num_trials):
            #reset cluster health state
            self.reset_cluster_health()
            
            #randomly flag DataNodes as failed based on the 5% node failure rate
            for node in node_names:
                if np.random.rand() < failure_rate:
                    self.datanodes[node]['status'] = 'failed'
            
            #detect Data Loss for each of the 1000 blocks
            #a block is lost only if all its replica-hosting nodes are marked as failed
            lost_blocks_count = 0
            for block_id, block_info in self.namenode['blocks'].items():
                active_replicas = [node for node in block_info['replicas'] 
                                   if self.datanodes[node]['status'] == 'active']
                
                #f active replica count hits zero, the entire block is considered LOST
                if len(active_replicas) == 0:
                    lost_blocks_count += 1
            
            # record total data damage stats for this specific simulation trial
            trials_lost_blocks.append(lost_blocks_count)
            
        return np.array(trials_lost_blocks)


# execution Pipeline matching Lab Manual guidelines (Page 10 style)

# 1.initialize HDFS Fault Tolerance Simulator with 10 nodes and RF = 3
hdfs_sim = HDFSFaultToleranceSimulator(num_nodes=10, replication_factor=3)

# 2.distribute the dataset containing 1,000 blocks across the cluster
hdfs_sim.place_dataset_blocks(num_blocks=1000)

# 3.execute 1,000 randomized simulation trials
losses_rf3 = hdfs_sim.simulate_disaster_scenarios(num_trials=1000, failure_rate=0.05)

#calculate summary metrics to confirm code output validity
total_times_lost = np.sum(losses_rf3 > 0)
print(" Simulation completed successfully following the lab manual's architecture structure.")
print(f"  Number of trials where the cluster lost at least one block: {total_times_lost} times out of 1000.")

#determine the dynamic range of bins based on the maximum loss detected in the simulation
max_loss_detected = np.max(losses_rf3)
bin_edges = np.arange(max(5, max_loss_detected) + 2) - 0.5

#create the figure and axis plotting structure matching Lab 4 manual specifications
fig, ax = plt.subplots(figsize=(8, 5))

#plot the histogram with clear borders, custom color, and specific width settings
ax.hist(losses_rf3, 
        bins=bin_edges, 
        color='#2ca02c', 
        edgecolor='black', 
        alpha=0.7, 
        rwidth=0.8)

# Set the titles and labels matching the exact assignment requirements
ax.set_title('HDFS Failure Simulation Histogram (Replication Factor = 3)', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Blocks Lost', fontsize=10)
ax.set_ylabel('Frequency (Out of 1000 Trials)', fontsize=10)

#configure gridlines and axis ticks for clear readability
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(range(max(5, max_loss_detected) + 2))

#adjust layout and save the visualization as an image file
plt.tight_layout()
plt.savefig('hdfs_failure_simulation_rf3.png')
plt.show()

print(" Histogram visualization generated and saved successfully as 'hdfs_failure_simulation_rf3.png'")